In [5]:
!pip install vpython
from vpython import *
import math as m

<IPython.core.display.Javascript object>

In [23]:
#Set up 1 (inicialización de datos)
a1=float(input("Angulo meridiano (greenwich es 0)"))
a2=float(input("Angulo latitud (polo norte +90, polo sur -90)"))
Delta_v= float(input("ingrese delta v inicial"))

Angulo meridiano (greenwich es 0) 0
Angulo latitud (polo norte +90, polo sur -90) 0
ingrese delta v inicial 8000


In [25]:
# Set up 2
#Creamos la escena 
scene = canvas()
#coordenadas cartesianas S a S'
def cart(r,alpha,beta): #Alpha= meridiano Beta=latitud respecto ecuador
    x=r*m.cos(alpha)*m.cos(beta)
    y=r*m.sin(alpha)*m.cos(beta)
    z=r*sin(beta)
    return x,y,z #en sistema tierra 
#Definimos la función de orientación de tiro (vertical en este caso)
def Orientación(DeltaV,Meridiano,Paralelo):
    V_x=DeltaV*m.cos(Paralelo)*m.cos(Meridiano)
    V_y=DeltaV*m.cos(Paralelo)*m.sin(Meridiano)
    V_z=DeltaV*m.sin(Paralelo)
    return V_x, V_y, V_z
def fuerzas_grav(p1, p2, m1, m2,G):
    rV = p1 - p2
    rM = mag(rV)
    unitar = rV / rM
    F = unitar * G * m1 * m2 / rM**2
    return F

#En esta parte se pondrán los inputs


#Definimos las coordenadas

#pasamos de angulos sexag a rad
a1=a1*m.pi/180
a2=a2*m.pi/180

#datos iniciales
r_1=6371000
G=6.67*10**-11
omega= 7.27*10**-5 #rad/s
#Definimos los diferenciales de tiempo 
t=0
dt=5 #15 min en seg 

#Ejes de comprobación (Por si se necesitan, entonces se quitan los asteriscos y se hace grande la bola blanca)
#Eje_x= box(position=vec(0,0,0),size=vec(3*r_1,r_1/10,r_1/10), color=color.red)
#Eje_y= box(position=vec(0,0,0),size=vec(r_1/10,3*r_1,r_1/10), color=color.yellow)
Eje_z= box(position=vec(0,0,0),size=vec(r_1/10,r_1/10,3*r_1), color=color.green)
#Definimos los cuerpos 

Planeta_1= sphere(position=vec(0,0,0),radius=r_1, color=color.blue)
masa_pl_1=5.97*10**24 #Kg
Planeta_1.momentum=vec(0,0,0)


xo_1,yo_1,zo_1=cart(r_1,a1,a2)
print(xo_1,yo_1,zo_1)
Objeto_1= sphere(pos=vec(xo_1,yo_1,zo_1),radius=r_1/100,make_trail=True)
masa_obj_1=10 #kg
Objeto_1.momentum= vec(0,0,0)
print(xo_1,yo_1,zo_1)


#Definimos primero el giro terrestre al principio 
Objeto_1.momentum_rotatingEarth = masa_obj_1 * cross(vec(0,0,omega), vec(xo_1,yo_1,zo_1))

#definimos el impulso inicial en base al delta v 
V_x, V_y, V_z=Orientación(Delta_v,a1,a2)
Objeto_1.initialMomentum=vec(V_x, V_y, V_z)*masa_obj_1
#Juntamos todos los momentos iniciales (Solo aplicados al inicio) 
Objeto_1.momentum = Objeto_1.momentum_rotatingEarth +Objeto_1.initialMomentum
print(V_x, V_y, V_z)
print("Momento inicial=",Objeto_1.momentum)

<IPython.core.display.Javascript object>

6371000.0 0.0 0.0
6371000.0 0.0 0.0
8000.0 0.0 0.0
Momento inicial= <80000, 4631.72, 0>


In [27]:

# Loop
detener=False
while t<6000:
    rate(50)
    for i in range(30):
        x_ob1act = Objeto_1.pos.x
        y_ob1act = Objeto_1.pos.y
        z_ob1act = Objeto_1.pos.z
        r=(x_ob1act**2+y_ob1act**2+z_ob1act**2)**(1/2)
        Objeto_1.momentum_gravity = fuerzas_grav(Planeta_1.pos,Objeto_1.pos,masa_pl_1,masa_obj_1,G) *dt
        Objeto_1.momentum += Objeto_1.momentum_gravity #Para añadir cpsas al momento aqui 
        Objeto_1.pos=Objeto_1.momentum *dt/masa_obj_1 +Objeto_1.pos #Esto está bien pero necesito una restricción de gravedad para que no se vaya volando 
        t += dt
        if r<r_1: #Funciona mas menos
            detener=True
            break
    if detener==True:
        break 
    #print(Objeto_1.momentum,Objeto_1.pos,Objeto_1.momentum_gravity) 
    print("r:", r, "momentum:", Objeto_1.momentum)
    print(t,"Posición",Objeto_1.pos)

r: 7435209.735356384 momentum: <67437.1, 4575.52, 0>
150 Posición <7.46863e+06, 69163.6, 0>
r: 8375682.988036878 momentum: <57913.4, 4454.97, 0>
300 Posición <8.40356e+06, 136910, 0>
r: 9187746.06338798 momentum: <50194, 4307.92, 0>
450 Posición <9.21066e+06, 202617, 0>
r: 9893358.0242377 momentum: <43652.3, 4148.72, 0>
600 Posición <9.91166e+06, 266012, 0>
r: 10507291.022118105 momentum: <37928.4, 3983.37, 0>
750 Posición <1.05212e+07, 326967, 0>
r: 11040015.61090802 momentum: <32796.7, 3814.47, 0>
900 Posición <1.10498e+07, 385412, 0>
r: 11499220.920607667 momentum: <28106.1, 3642.98, 0>
1050 Posición <1.15049e+07, 441303, 0>
r: 11890686.795218213 momentum: <23750.2, 3469.09, 0>
1200 Posición <1.18923e+07, 494603, 0>
r: 12218816.349528654 momentum: <19650.2, 3292.54, 0>
1350 Posición <1.22165e+07, 545275, 0>
r: 12486976.43030415 momentum: <15745.4, 3112.78, 0>
1500 Posición <1.24808e+07, 593274, 0>
r: 12697722.367559126 momentum: <11987.7, 2929.08, 0>
1650 Posición <1.26877e+07, 6385

Este pequeño codigo muestra el tiro inicial de un cohete sin corregir la trayectoria de este. \
El cohete ejecuta su quema en un tiempo ideal en un instante y con progrado hacia el componente vetical.\
Al subir, como la tierra rota provoca que el cohete salga con un a velocidad inicial equivalente a la velocidad de la rotación de la tierra en la latitud en la que se encuentre, lo que genera una trayectoria parabolica